In [1]:
# imports
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as plt
import tensorflow as tf
import random
from sklearn.model_selection import KFold
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical

In [2]:
import tensorflow as tf

print("TF Version:", tf.__version__)
print("Devices:", tf.config.list_physical_devices())

TF Version: 2.10.0
Devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [3]:
class_df = pd.read_csv("E:/Datasets/AIA-image-classification/images.csv")
folder = "E:/Datasets/AIA-image-classification/images"
class_df

,image_name,label
0,0.jpg,0
1,1.jpg,4
2,2.jpg,5
3,4.jpg,0
4,7.jpg,4
...,...,...
14029,20048.jpg,0
14030,20051.jpg,1
14031,20052.jpg,3
14032,20053.jpg,4


In [4]:
image_paths = []
missing_files = []
for index, row in class_df.iterrows():
    image_id = str(row['image_name'])
   
    full_path = os.path.join(folder, image_id)
    if os.path.exists(full_path):
        image_paths.append(full_path)
    else:
        image_paths.append(None)
        missing_files.append(image_id)

class_df['image_paths'] = image_paths

class_df = class_df.dropna(subset=['image_paths'])

df = class_df.drop(columns=['image_name'])

df['label'] = df['label'].astype(str)

df.to_csv("E:/Datasets/AIA-image-classification.csv", index=False)

print("Missing files:", missing_files)
print(df.info())
df

Missing files: []
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14034 entries, 0 to 14033
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   label        14034 non-null  object
 1   image_paths  14034 non-null  object
dtypes: object(2)
memory usage: 219.4+ KB
None


,label,image_paths
0,0,E:/Datasets/AIA-image-classification/images\0.jpg
1,4,E:/Datasets/AIA-image-classification/images\1.jpg
2,5,E:/Datasets/AIA-image-classification/images\2.jpg
3,0,E:/Datasets/AIA-image-classification/images\4.jpg
4,4,E:/Datasets/AIA-image-classification/images\7.jpg
...,...,...
14029,0,E:/Datasets/AIA-image-classification/images\20...
14030,1,E:/Datasets/AIA-image-classification/images\20...
14031,3,E:/Datasets/AIA-image-classification/images\20...
14032,4,E:/Datasets/AIA-image-classification/images\20...


In [5]:
from PIL import Image

bad_images = []
for image in df['image_paths']:
    try:
        img = Image.open(image)  
        img.verify()             
    except:
        bad_images.append(image)

len(bad_images)

0

In [6]:
# input split
from sklearn.model_selection import train_test_split
train, test = train_test_split(df, test_size=0.2, random_state=42)

In [7]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_generator = ImageDataGenerator(
    rescale = 1./255,
    rotation_range=40,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True)
val_generator = ImageDataGenerator(rescale=1./255)

In [8]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.metrics import Precision, Recall, Accuracy

def create_model():
    model = Sequential([
        Conv2D(16, (3, 3), activation='relu', input_shape=(256, 256, 3)),
        MaxPooling2D(),
        BatchNormalization(),
        
        Conv2D(32, (3, 3), activation='relu'),
        MaxPooling2D(),
        BatchNormalization(),
    
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D(),
        BatchNormalization(),
    
        Conv2D(128, (3, 3), activation='relu'),
        MaxPooling2D(),
        BatchNormalization(),
        
        Conv2D(256, (3, 3), activation='relu'),
        MaxPooling2D(),
        BatchNormalization(),
    
        Flatten(),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(6, activation='softmax')    
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy', Precision(), Recall()])
    return model

model = create_model()
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 254, 254, 16)      448       
                                                                 
 max_pooling2d (MaxPooling2D  (None, 127, 127, 16)     0         
 )                                                               
                                                                 
 batch_normalization (BatchN  (None, 127, 127, 16)     64        
 ormalization)                                                   
                                                                 
 conv2d_1 (Conv2D)           (None, 125, 125, 32)      4640      
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 62, 62, 32)       0         
 2D)                                                             
                                                        

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(patience=5, restore_best_weights=True)

In [ ]:
from sklearn.model_selection import KFold

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

acc_scores = []
precision_scores = []
recall_scores = []

for fold, (train_idx, val_idx) in enumerate(kfold.split(df)):

    print(f"\n🔥 Fold {fold+1}")

    # Split dataframe
    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]

    # Generators
    train_iterator = train_generator.flow_from_dataframe(
    train_df,
    x_col='image_paths',
    y_col='label',
    target_size=(256, 256),
    batch_size=256,
    class_mode='categorical')

    val_iterator = val_generator.flow_from_dataframe(
        val_df,
        x_col='image_paths',
        y_col='label',
        target_size=(256, 256),
        batch_size=256,
        class_mode='categorical'
    )
    
    model = create_model()

    # Train
    model.fit(
        train_iterator,
        validation_data=val_iterator,
        epochs=20,
        callbacks=[early_stop])

    # Evaluate
    loss, acc, precision, recall = model.evaluate(val_iterator)
    print("Accuracy:", acc)
    # Store scores
    acc_scores.append(acc)
    precision_scores.append(precision)
    recall_scores.append(recall)
print('🔥 Done !')


🔥 Fold 1
Found 11227 validated image filenames belonging to 6 classes.
Found 2807 validated image filenames belonging to 6 classes.
Epoch 1/20
44/44 [==============================] - 80s 2s/step - loss: 1.9054 - accuracy: 0.5285 - precision_2: 0.5707 - recall_2: 0.4670 - val_loss: 4.1505 - val_accuracy: 0.1835 - val_precision_2: 0.1774 - val_recall_2: 0.1646
Epoch 2/20
19/44 [===========>..................] - ETA: 41s - loss: 1.3264 - accuracy: 0.6076 - precision_2: 0.6611 - recall_2: 0.5254

In [ ]:
# Final result
print("🔥 Average Accuracy:", sum(acc_scores)/len(acc_scores))
print("🔥 Precision:", precision_scores)
print("🔥 Recall:", recall_scores)